In [1]:
import os
import torch
import numpy as np
import cv2

from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = "/Users/arnavkarnik/Desktop/AIBD_Project/Aishwarya thesis"

# 🔥 UPDATED: Include NEGATIVES as class 0
CLASSES = ["NEGATIVES", "PV", "PF", "BP", "EBA", "Vasculitis"]

# 🔥 EXPLICIT mapping (DO NOT rely on enumerate order later)
CLASS_TO_IDX = {
    "NEGATIVES": 0,
    "PV": 1,
    "PF": 2,
    "BP": 3,
    "EBA": 4,
    "Vasculitis": 5
}

NUM_CLASSES = len(CLASSES)

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20

Using device: cpu


In [2]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    
    # 🔥 ADD THIS (important for stable training across NEGATIVE vs disease)
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    
    # 🔥 MUST match train normalization
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

In [3]:
class AIBDDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for cls in CLASSES:
            cls_path = os.path.join(root_dir, cls)
            if not os.path.exists(cls_path):
                continue

            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                    self.samples.append((
                        os.path.join(cls_path, img_name),
                        CLASS_TO_IDX[cls]
                    ))

        print("Total samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def extract_bmz_profile(self, image):
        gray = image.mean(dim=0)
        H = gray.shape[0]

        top = gray[:H//3, :].mean()
        middle = gray[H//3:2*H//3, :].mean()
        bottom = gray[2*H//3:, :].mean()

        return torch.tensor([top, middle, bottom], dtype=torch.float32)

    # 🔥 UPDATED: robust BMZ mask (handles NEGATIVES)
    def extract_bmz_mask(self, image):
        gray = image.mean(dim=0).cpu().numpy()

        grad = np.abs(np.gradient(gray, axis=0))
        grad = cv2.GaussianBlur(grad, (9, 9), 0)

        # 🔥 FIX: if gradient is too weak → no BMZ signal (NEGATIVE case)
        if np.max(grad) < 0.05:
            mask = np.zeros_like(gray)
            return torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        bmz_line = np.argmax(grad, axis=0)

        mask = np.zeros_like(gray)

        for x in range(len(bmz_line)):
            y = bmz_line[x]

            y1 = max(0, y - 4)
            y2 = min(mask.shape[0], y + 4)

            mask[y1:y2, x] = 1

        mask = torch.tensor(mask, dtype=torch.float32)
        return mask.unsqueeze(0)

    # 🔥 UPDATED: stable distance map
    def compute_distance_map(self, bmz_mask):
        mask = bmz_mask.squeeze(0).cpu().numpy()

        # 🔥 FIX: if mask is empty (NEGATIVE case)
        if np.sum(mask) < 10:
            return torch.zeros_like(bmz_mask)

        dist_out = cv2.distanceTransform((1 - mask).astype(np.uint8), cv2.DIST_L2, 5)
        dist_in = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5)

        signed_dist = dist_out - dist_in

        signed_dist = signed_dist / (np.max(np.abs(signed_dist)) + 1e-6)

        return torch.tensor(signed_dist, dtype=torch.float32).unsqueeze(0)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        profile = self.extract_bmz_profile(image)
        bmz_mask = self.extract_bmz_mask(image)
        distance_map = self.compute_distance_map(bmz_mask)

        return {
            "image": image,
            "bmz_mask": bmz_mask,
            "distance_map": distance_map,
            "profile": profile,
            "label": torch.tensor(label, dtype=torch.long)
        }

In [4]:
dataset = AIBDDataset(DATA_DIR, transform=None)  # 🔥 NO transform here

indices = list(range(len(dataset)))
labels = [dataset.samples[i][1] for i in indices]

# 🔥 DEBUG: class distribution check (VERY IMPORTANT NOW)
from collections import Counter
print("Full dataset distribution:", Counter(labels))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# 🔥 Apply transforms AFTER split
train_dataset = Subset(AIBDDataset(DATA_DIR, transform=train_transform), train_idx)
val_dataset = Subset(AIBDDataset(DATA_DIR, transform=val_transform), val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train:", len(train_dataset), "Val:", len(val_dataset))

Total samples: 7883
Full dataset distribution: Counter({3: 2937, 1: 2719, 5: 1488, 0: 378, 4: 197, 2: 164})
Total samples: 7883
Total samples: 7883
Train: 6306 Val: 1577


In [5]:
class BMZAttention(nn.Module):
    def __init__(self, in_channels=512):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, 256, 1),
            nn.ReLU(),
            nn.Conv2d(256, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attn = self.attention(x)

        # 🔥 NEW: clamp attention (prevents spikes on noisy NEGATIVE samples)
        attn = torch.clamp(attn, 0.0, 1.0)

        # 🔥 STABILITY: controlled amplification
        out = x * (1 + 0.8 * attn)

        return out, attn

In [6]:
class ImageEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # 🔥 CHANGE: 3 → 4 input channels
        backbone.conv1 = nn.Conv2d(
            4, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.attn = BMZAttention(512)

        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, out_dim)
        )

    def forward(self, x, bmz_mask, distance_map):
        # 🔥 Ensure distance_map is safe (NEGATIVE stability)
        distance_map = torch.clamp(distance_map, -1.0, 1.0)

        # 🔥 ADD distance channel
        x = torch.cat([x, distance_map], dim=1)

        feat_map = self.features(x)

        bmz_mask = nn.functional.interpolate(
            bmz_mask,
            size=feat_map.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        # 🔥 NEW: prevent BMZ amplification on empty masks (NEGATIVE case)
        bmz_strength = torch.mean(bmz_mask, dim=[1,2,3], keepdim=True)
        bmz_gate = (bmz_strength > 0.01).float()

        feat_map = feat_map * (1 + 0.8 * bmz_mask * bmz_gate)

        feat_map, attn = self.attn(feat_map)

        pooled = self.pool(feat_map).view(x.size(0), -1)
        out = self.projection(pooled)

        return out, attn

In [7]:
class BMZProfileEncoder(nn.Module):
    def __init__(self, input_dim=3, out_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),

            # 🔥 UPDATED: more stable for small/imbalanced batches
            nn.BatchNorm1d(32, eps=1e-5, momentum=0.1),

            nn.Linear(32, out_dim)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
class PrototypeLayer(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=320):
        super().__init__()

        self.prototypes = nn.Parameter(torch.randn(num_classes, feat_dim))

        # 🔥 IMPORTANT: initialize properly (helps rare classes like PF/EBA)
        nn.init.kaiming_normal_(self.prototypes, nonlinearity='linear')

        self.scale = nn.Parameter(torch.tensor(10.0))

    def forward(self, features):
        features = nn.functional.normalize(features, dim=1)
        prototypes = nn.functional.normalize(self.prototypes, dim=1)

        logits = torch.matmul(features, prototypes.T)

        return logits * self.scale

In [9]:
class BCPLModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.image_encoder = ImageEncoder(256)
        self.profile_encoder = BMZProfileEncoder(3, 64)

        # 🔥 FIX: use NUM_CLASSES (ensures consistency everywhere)
        self.prototype_layer = PrototypeLayer(NUM_CLASSES, 320)

    def forward(self, image, bmz_mask, distance_map, profile):
        f_img, attn = self.image_encoder(image, bmz_mask, distance_map)
        f_prof = self.profile_encoder(profile)

        # 🔥 normalization (important for prototype stability)
        f_img = nn.functional.normalize(f_img, dim=1)
        f_prof = nn.functional.normalize(f_prof, dim=1)

        fused = torch.cat([f_img, f_prof], dim=1)

        # 🔥 NEW: normalize fused embedding (helps rare classes + NEGATIVE)
        fused = nn.functional.normalize(fused, dim=1)

        logits = self.prototype_layer(fused)

        return logits, fused, attn

In [10]:
labels_all = [dataset.samples[i][1] for i in range(len(dataset))]
class_counts = np.bincount(labels_all, minlength=NUM_CLASSES)

# 🔥 FIX 1: smoother inverse frequency (prevents extreme weights)
weights = 1.0 / (class_counts + 1e-6)
weights = weights / weights.sum() * NUM_CLASSES  # normalize

# 🔥 FIX 2: controlled boosting for hard classes
weights[CLASS_TO_IDX["PF"]] *= 1.5
weights[CLASS_TO_IDX["EBA"]] *= 1.3

# 🔥 FIX 3: ensure NEGATIVE is not ignored
weights[CLASS_TO_IDX["NEGATIVES"]] *= 1.2

weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", weights)


# 🔥 FOCAL LOSS (stable)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()


ce_loss_fn = FocalLoss(alpha=weights, gamma=2)


def prototype_loss(features, labels, prototypes):
    proto = prototypes[labels]
    return ((features - proto) ** 2).sum(dim=1).mean()


# 🔥 UPDATED: attention loss safer for NEGATIVES
def attention_loss(attn):
    sparsity = attn.mean()
    sharpness = (attn * (1 - attn)).mean()

    # 🔥 reduce penalty slightly (NEGATIVE images naturally flat)
    return 0.7 * sparsity + 0.3 * sharpness


def compute_loss(logits, features, labels, model, attn):
    ce = ce_loss_fn(logits, labels)

    proto = prototype_loss(
        features,
        labels,
        model.prototype_layer.prototypes
    )

    attn_l = attention_loss(attn)

    # 🔥 FINAL BALANCE (important)
    total = ce + 0.2 * proto + 0.1 * attn_l

    return total, ce, proto, attn_l

Class counts: [ 378 2719  164 2937  197 1488]
Class weights: tensor([1.2532, 0.1452, 3.6105, 0.1344, 2.6049, 0.2653])


In [12]:
# =========================
# 🧪 ABLATION 1: BASELINE (IMAGE ONLY) — FIXED
# =========================

model = BCPLModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

from sklearn.metrics import accuracy_score, classification_report

def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)

            # ZERO BMZ + DIST
            bmz_mask = torch.zeros_like(batch["bmz_mask"]).to(DEVICE)
            distance_map = torch.zeros_like(batch["distance_map"]).to(DEVICE)

            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return accuracy_score(labels_all, preds_all)


def evaluate_detailed(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)

            bmz_mask = torch.zeros_like(batch["bmz_mask"]).to(DEVICE)
            distance_map = torch.zeros_like(batch["distance_map"]).to(DEVICE)

            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(labels_all, preds_all, target_names=CLASSES))


# ===== TRAIN =====
for epoch in range(10):
    model.train()

    for batch in tqdm(train_loader):
        images = batch["image"].to(DEVICE)

        bmz_mask = torch.zeros_like(batch["bmz_mask"]).to(DEVICE)
        distance_map = torch.zeros_like(batch["distance_map"]).to(DEVICE)

        profile = batch["profile"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        logits, features, attn = model(images, bmz_mask, distance_map, profile)

        loss, _, _, _ = compute_loss(logits, features, labels, model, attn)

        loss.backward()
        optimizer.step()

# ===== EVAL =====
val_acc = evaluate(val_loader)
print("✅ BASELINE VAL ACC:", val_acc)
evaluate_detailed(val_loader)

100%|██████████| 395/395 [07:27<00:00,  1.13s/it]


✅ BASELINE VAL ACC: 0.29613189600507295

📊 Classification Report:

              precision    recall  f1-score   support

   NEGATIVES       0.53      0.95      0.68        76
          PV       0.43      0.02      0.03       544
          PF       0.06      0.91      0.11        33
          BP       0.88      0.28      0.43       587
         EBA       0.07      0.92      0.13        39
  Vasculitis       0.86      0.52      0.65       298

    accuracy                           0.30      1577
   macro avg       0.47      0.60      0.34      1577
weighted avg       0.67      0.30      0.33      1577



In [13]:
# =========================
# 🧪 ABLATION 2: + BMZ (FIXED)
# =========================

model = BCPLModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

from sklearn.metrics import accuracy_score, classification_report

def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)

            # 🔥 ZERO DISTANCE
            distance_map = torch.zeros_like(batch["distance_map"]).to(DEVICE)

            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return accuracy_score(labels_all, preds_all)


def evaluate_detailed(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = torch.zeros_like(batch["distance_map"]).to(DEVICE)

            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(labels_all, preds_all, target_names=CLASSES))


# ===== TRAIN =====
for epoch in range(10):
    model.train()

    for batch in tqdm(train_loader):
        images = batch["image"].to(DEVICE)
        bmz_mask = batch["bmz_mask"].to(DEVICE)

        distance_map = torch.zeros_like(batch["distance_map"]).to(DEVICE)

        profile = batch["profile"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        logits, features, attn = model(images, bmz_mask, distance_map, profile)

        loss, _, _, _ = compute_loss(logits, features, labels, model, attn)

        loss.backward()
        optimizer.step()

# ===== EVAL =====
val_acc = evaluate(val_loader)
print("✅ +BMZ VAL ACC:", val_acc)
evaluate_detailed(val_loader)

100%|██████████| 395/395 [07:37<00:00,  1.16s/it]


✅ +BMZ VAL ACC: 0.3417882054533925

📊 Classification Report:

              precision    recall  f1-score   support

   NEGATIVES       0.54      0.97      0.70        76
          PV       0.85      0.04      0.08       544
          PF       0.07      0.94      0.13        33
          BP       0.89      0.19      0.31       587
         EBA       0.08      0.90      0.14        39
  Vasculitis       0.68      0.90      0.77       298

    accuracy                           0.34      1577
   macro avg       0.52      0.66      0.36      1577
weighted avg       0.78      0.34      0.33      1577



In [14]:
# =========================
# 🧪 ABLATION 3: + DISTANCE (FIXED)
# =========================

model = BCPLModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

from sklearn.metrics import accuracy_score, classification_report

def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)

            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return accuracy_score(labels_all, preds_all)


def evaluate_detailed(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)

            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(labels_all, preds_all, target_names=CLASSES))


# ===== TRAIN =====
for epoch in range(10):
    model.train()

    for batch in tqdm(train_loader):
        images = batch["image"].to(DEVICE)
        bmz_mask = batch["bmz_mask"].to(DEVICE)
        distance_map = batch["distance_map"].to(DEVICE)

        profile = batch["profile"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        logits, features, attn = model(images, bmz_mask, distance_map, profile)

        loss, _, _, _ = compute_loss(logits, features, labels, model, attn)

        loss.backward()
        optimizer.step()

# ===== EVAL =====
val_acc = evaluate(val_loader)
print("✅ +DISTANCE VAL ACC:", val_acc)
evaluate_detailed(val_loader)

100%|██████████| 395/395 [07:54<00:00,  1.20s/it]


✅ +DISTANCE VAL ACC: 0.2872542802790108

📊 Classification Report:

              precision    recall  f1-score   support

   NEGATIVES       0.37      0.97      0.54        76
          PV       0.87      0.02      0.05       544
          PF       0.08      0.94      0.14        33
          BP       0.71      0.09      0.17       587
         EBA       0.07      0.95      0.13        39
  Vasculitis       0.69      0.82      0.75       298

    accuracy                           0.29      1577
   macro avg       0.46      0.63      0.30      1577
weighted avg       0.71      0.29      0.25      1577



In [15]:
# =========================
# 🧪 ABLATION 4: FULL MODEL (FIXED)
# =========================

model = BCPLModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

from sklearn.metrics import accuracy_score, classification_report

def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)
            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return accuracy_score(labels_all, preds_all)


def evaluate_detailed(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)
            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(labels_all, preds_all, target_names=CLASSES))


# ===== TRAIN =====
for epoch in range(10):
    model.train()

    for batch in tqdm(train_loader):
        images = batch["image"].to(DEVICE)
        bmz_mask = batch["bmz_mask"].to(DEVICE)
        distance_map = batch["distance_map"].to(DEVICE)
        profile = batch["profile"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        logits, features, attn = model(images, bmz_mask, distance_map, profile)

        loss, _, _, _ = compute_loss(logits, features, labels, model, attn)

        loss.backward()
        optimizer.step()

# ===== EVAL =====
val_acc = evaluate(val_loader)
print("✅ FULL MODEL VAL ACC:", val_acc)
evaluate_detailed(val_loader)

100%|██████████| 395/395 [10:06<00:00,  1.54s/it]


✅ FULL MODEL VAL ACC: 0.2352568167406468

📊 Classification Report:

              precision    recall  f1-score   support

   NEGATIVES       0.72      0.91      0.80        76
          PV       0.50      0.00      0.01       544
          PF       0.05      0.85      0.09        33
          BP       0.80      0.03      0.05       587
         EBA       0.06      0.97      0.11        39
  Vasculitis       0.88      0.73      0.80       298

    accuracy                           0.24      1577
   macro avg       0.50      0.58      0.31      1577
weighted avg       0.67      0.24      0.22      1577

